# Table model vs ResNet-UNet vs graph networks on the 11 shared markers (Jackson → Danenberg)

**Question.** For imputing a missing marker per cell, does a graph neural network (cell graph, message passing) beat:
- the per-cell table model;
- the image model (your ResNet-UNet checkpoints)?

**Common setting** (as in `kill-test`):
- Train on Jackson 2020; test on Danenberg 2022 cells in tumour images, with no fine-tuning.
- For each of the 11 shared markers, the inputs are the other 10 shared markers + H3, DNA1, DNA2 + log cell area.
- Danenberg cell inputs are averaged from the images over the published masks. Jackson inputs come from its cell table.
- Inputs are harmonised per cohort (median, p5–p95).

**Models** (one per target marker):
- **Table:** LightGBM on the cell's own inputs.
- **Table + ring:** LightGBM on the same, plus the mean inputs of non-touching cells within 30 µm (`kill-test` B2).
- **MLP:** 2 hidden layers of 128, on the cell's own inputs. This is the graph models without the graph.
- **GCN:** Kipf GCN, 2 layers of 128, with the symmetric normalised adjacency including self-loops.
- **GraphSAGE:** 2 layers of 128, with separate weights for the cell itself and for the mean of its neighbours.
- **Graph:** the 10 nearest cells by centroid within the same image, symmetrised. The same rule in both cohorts, because their published contact graphs were built differently.
- **Training:** full-batch on the whole Jackson graph. MSE on the standardised arcsinh(x/5) target, Adam (lr 3e-3), and early stopping on the validation patients.
- **ResNet-UNet:** the three existing checkpoints (residual, gamma, immuvis). Their per-cell predictions are reused from `kill-test`.

**Jackson split:** patients 70 / 10 / 20 (train / validation for early stopping / held-out test).
- Every Jackson-trained model (table, MLP, GCN, GraphSAGE) is fitted on the same 70%.
- The ResNet-UNet was trained on Jackson images earlier, so it cannot be scored fairly within Jackson. It is scored on Danenberg only.

**Readouts, per marker:**
- per-cell r;
- R4: per-cell r after removing each image's mean;
- 1000× patient-bootstrap CIs and paired differences between models.
- **Danenberg truth:** the per-cell mean of the model's target map (`kill-test`).
- **Caveat:** ERα saturates in that space (different ER antibodies in the two cohorts). ERα is reported but not used for conclusions.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Reuse `kill-test` per-cell truth and ResNet-UNet predictions; recompute Danenberg per-cell image means

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
RESNET = {(a, m): KT[f"pred__{a}__{m}"].values.astype(np.float32) for a in ARMS for m in MARKERS}
del KT

tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def cell_mean(flat, pm):
    v, loc, n, _ = pm
    cnt = np.bincount(loc, minlength=n)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, np.bincount(loc, flat[v], n) / np.maximum(cnt, 1), np.nan).astype(np.float32)

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    pm = (v, loc[v], len(obj), m.shape)
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    return np.stack([cell_mean(im[:, c], pm) for c in range(im.shape[1])], 1)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(IMGS, ST[:-1], ST[1:])))
assert RAW.shape == (N, len(D_CHAN))
tick("kill-test outputs loaded; per-cell image means recomputed")

## Features, graphs and the Jackson patient split

In [ ]:
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch
import torch.nn as nn
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
K_NN = 10
EPOCHS = 30 if SMOKE else 400
PATIENCE = 10 if SMOKE else 40
HID = 32 if SMOKE else 128

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
XD_in = np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF).astype(np.float32)
XJ_in = np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF).astype(np.float32)
HD = np.c_[harm(XD_in), harm(DD["log_area"][:, None])]
HJ = np.c_[harm(XJ_in), harm(DJ["log_area"][:, None])]
AREA = len(IN_SET)                                   # column index of log area in HD / HJ
MrD, nrD = ring_means(HD[:, :AREA], DD, contact_matrix(DD))
MrJ, nrJ = ring_means(HJ[:, :AREA], DJ, contact_matrix(DJ))

def knn_graph(D):
    n = len(D["cells"])
    rows, cols = [], []
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        k = min(K_NN, b - a - 1)
        _, nb = cKDTree(D["xy"][a:b]).query(D["xy"][a:b], k=k + 1)
        rows.append(np.repeat(np.arange(a, b), k))
        cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(n, n)).tocsr()
    A = ((A + A.T) > 0).astype(np.float32)
    deg = np.asarray(A.sum(1)).ravel()
    A_mean = sp.diags(1 / np.maximum(deg, 1)) @ A                       # GraphSAGE mean over neighbours
    Ai = A + sp.eye(n, dtype=np.float32)
    dh = 1 / np.sqrt(np.asarray(Ai.sum(1)).ravel())
    A_gcn = sp.diags(dh) @ Ai @ sp.diags(dh)                            # Kipf normalisation with self-loops
    return A_mean.tocoo(), A_gcn.tocoo(), float(deg.mean())

def to_t(M):
    M = M.tocoo()
    i = torch.from_numpy(np.vstack([M.row, M.col]).astype(np.int64))
    return torch.sparse_coo_tensor(i, torch.from_numpy(M.data.astype(np.float32)), M.shape).coalesce().to(dev)

AmJ, AgJ, degJ = knn_graph(DJ)
AmD, AgD, degD = knn_graph(DD)
G = {"jackson": {"mean": to_t(AmJ), "gcn": to_t(AgJ)}, "danenberg": {"mean": to_t(AmD), "gcn": to_t(AgD)}}
print(f"kNN graphs: mean degree Jackson {degJ:.1f}, Danenberg {degD:.1f}; device {dev}")

REP = 0

## Models

In [ ]:
class MLP(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h), nn.ReLU(), nn.Dropout(p),
                              nn.Linear(h, 1))
    def forward(s, x, A):
        return s.net(x).squeeze(-1)

class GCN(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.l1, s.l2, s.out, s.dp = nn.Linear(d, h), nn.Linear(h, h), nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, A):
        h = s.dp(torch.relu(torch.sparse.mm(A, s.l1(x))))
        h = torch.relu(torch.sparse.mm(A, s.l2(h)))
        return s.out(h).squeeze(-1)

class SAGE(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.s1, s.n1 = nn.Linear(d, h), nn.Linear(d, h, bias=False)
        s.s2, s.n2 = nn.Linear(h, h), nn.Linear(h, h, bias=False)
        s.out, s.dp = nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, A):
        h = s.dp(torch.relu(s.s1(x) + torch.sparse.mm(A, s.n1(x))))
        h = torch.relu(s.s2(h) + torch.sparse.mm(A, s.n2(h)))
        return s.out(h).squeeze(-1)

NETS = {"MLP": (MLP, "mean"), "GCN": (GCN, "gcn"), "GraphSAGE": (SAGE, "mean")}

def train_net(cls, X, A, y):
    torch.manual_seed(SEED + REP)
    m = cls(X.shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3, weight_decay=1e-5)
    tr, va = torch.from_numpy(TR).to(dev), torch.from_numpy(VA).to(dev)
    best, state, bad, ep = np.inf, None, 0, 0
    for ep in range(EPOCHS):
        m.train()
        opt.zero_grad()
        loss = ((m(X, A)[tr] - y[tr]) ** 2).mean()
        loss.backward()
        opt.step()
        if ep % 5 == 4:
            m.eval()
            with torch.no_grad():
                v = ((m(X, A)[va] - y[va]) ** 2).mean().item()
            if v < best - 1e-5:
                best, bad = v, 0
                state = {k: t.detach().clone() for k, t in m.state_dict().items()}
            else:
                bad += 5
                if bad >= PATIENCE:
                    break
    m.load_state_dict(state)
    m.eval()
    return m, ep + 1, best

## Replicates: new seed and new Jackson patient split each time (replicate 0 = the first run)

In [ ]:
REPS = [int(r) for r in os.environ.get('REPS', '1,2').split(',')]
for REP in REPS:
    tick(f'replicate {REP}')
    jp, jcode = np.unique(DJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    TR, VA, TE = (np.flatnonzero(J_SPLIT == s) for s in ("train", "val", "test"))
    print({s: int((J_SPLIT == s).sum()) for s in ("train", "val", "test")}, "Jackson cells;",
          {s: int(len(set(DJ['patient'][J_SPLIT == s]))) for s in ("train", "val", "test")}, "patients")
    tick("features and graphs")

    PJ, PD, LOG = {}, {}, []
    for m in MARKERS:
        k = IN_SET.index(m)
        cols = [i for i in range(HJ.shape[1]) if i != k]                   # other 13 channels + log area
        yJ = XJ_in[:, k]
        mu, sd = yJ[TR].mean(), yJ[TR].std() + 1e-6
        # table models (LightGBM; NaN allowed)
        trs = np.sort(np.random.default_rng(SEED + REP).choice(TR, min(N_FIT_FINAL, len(TR)), replace=False))
        for name, XJm, XDm in (("Table", HJ[:, cols], HD[:, cols]),
                               ("Table + ring", np.c_[HJ[:, cols], MrJ[:, [c for c in cols if c < AREA]], nrJ],
                                np.c_[HD[:, cols], MrD[:, [c for c in cols if c < AREA]], nrD])):
            mdl = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(XJm[trs], yJ[trs])
            PJ[(name, m)], PD[(name, m)] = mdl.predict(XJm).astype(np.float32), mdl.predict(XDm).astype(np.float32)
        # neural models (NaN -> 0, the cohort median after harmonisation)
        XJt = torch.from_numpy(np.nan_to_num(HJ[:, cols])).to(dev)
        XDt = torch.from_numpy(np.nan_to_num(HD[:, cols])).to(dev)
        yt = torch.from_numpy((yJ - mu) / sd).to(dev)
        for name, (cls, akey) in NETS.items():
            net, n_ep, best = train_net(cls, XJt, G["jackson"][akey], yt)
            with torch.no_grad():
                PJ[(name, m)] = net(XJt, G["jackson"][akey]).cpu().numpy() * sd + mu
                PD[(name, m)] = net(XDt, G["danenberg"][akey]).cpu().numpy() * sd + mu
            LOG.append(dict(marker=m, model=name, epochs=n_ep, best_val_mse_std=best))
            del net
            if dev == "cuda":
                torch.cuda.empty_cache()
        tick(f"trained {m}")
    pd.DataFrame(LOG).to_csv(f"{OUT}/training_log_rep{REP}.csv", index=False)
    for a in ARMS:
        for m in MARKERS:
            PD[(f"ResNet-UNet {a}", m)] = RESNET[(a, m)]

    def stats_fn(img, pcode, n_pat, W, center):
        def f(a, b):
            ok = np.isfinite(a) & np.isfinite(b)
            a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), img[ok], pcode[ok]
            if center:
                k = int(img.max()) + 1
                cnt = np.maximum(np.bincount(im, minlength=k), 1)
                a = a - (np.bincount(im, a, k) / cnt)[im]
                b = b - (np.bincount(im, b, k) / cnt)[im]
            else:
                a, b = a - a.mean(), b - b.mean()
            sxy, sxx, syy = (np.bincount(pc, a * b, n_pat), np.bincount(pc, a * a, n_pat), np.bincount(pc, b * b, n_pat))
            return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))
        return f

    def ci(v):
        v = np.asarray(v)[np.isfinite(v)]
        return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

    # Danenberg: all analysed cells; truth = per-cell target map (kill-test)
    SCORE_D = {"R4": stats_fn(DD["img"], pcode, len(pats), W, True), "r": stats_fn(DD["img"], pcode, len(pats), W, False)}
    # Jackson: held-out test patients; truth = table value
    tp, tcode = np.unique(DJ["patient"][TE], return_inverse=True)
    WJ = np.random.default_rng(SEED + 12).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    SCORE_J = {"R4": stats_fn(DJ["img"][TE], tcode, len(tp), WJ, True), "r": stats_fn(DJ["img"][TE], tcode, len(tp), WJ, False)}
    ROWS, BOOT = [], {}
    for m in MARKERS:
        yJ = XJ_in[:, IN_SET.index(m)]
        for (name, mm), p in PJ.items():
            if mm != m:
                continue
            for sc, f in SCORE_J.items():
                pt, bv = f(yJ[TE], p[TE])
                BOOT[("jackson", m, name, sc)] = (pt, bv)
                ROWS.append(dict(cohort="Jackson (held-out patients)", marker=m, model=name, score=sc, value=pt,
                                 lo=ci(bv)[0], hi=ci(bv)[1]))
        for (name, mm), p in PD.items():
            if mm != m:
                continue
            for sc, f in SCORE_D.items():
                pt, bv = f(TRUTH[m], p)
                BOOT[("danenberg", m, name, sc)] = (pt, bv)
                ROWS.append(dict(cohort="Danenberg (cross-cohort)", marker=m, model=name, score=sc, value=pt,
                                 lo=ci(bv)[0], hi=ci(bv)[1]))
    RES = pd.DataFrame(ROWS)
    RES.to_csv(f"{OUT}/scores_rep{REP}.csv", index=False)
    ORDER = ["Table", "Table + ring", "MLP", "GCN", "GraphSAGE"] + [f"ResNet-UNet {a}" for a in ARMS]
    for coh in RES.cohort.unique():
        t = RES[(RES.cohort == coh) & (RES.score == "R4")].pivot(index="marker", columns="model", values="value")
        print(f"--- {coh}: R4"); print(t[[c for c in ORDER if c in t]].round(3).to_string())

    DIFF = []
    for coh in ("jackson", "danenberg"):
        models = [o for o in ORDER if (coh, MARKERS[0], o, "R4") in BOOT]
        for m in MARKERS:
            for a in models:
                for b in ("Table", "MLP"):
                    if a == b or (coh, m, a, "R4") not in BOOT:
                        continue
                    p1, b1 = BOOT[(coh, m, a, "R4")]
                    p2, b2 = BOOT[(coh, m, b, "R4")]
                    lo, hi = ci(b1 - b2)
                    DIFF.append(dict(cohort=coh, marker=m, model=a, vs=b, diff=p1 - p2, lo=lo, hi=hi,
                                     call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
    DIFF = pd.DataFrame(DIFF)
    DIFF.to_csv(f"{OUT}/differences_R4_rep{REP}.csv", index=False)
    SUMM = (DIFF[DIFF.marker != "estrogen_receptor_alpha"].groupby(["cohort", "model", "vs"]).call
            .value_counts().unstack(fill_value=0).reset_index())
    print(SUMM.to_string(index=False))
    tick(f'replicate {REP} done')